# 📄 PDF Table Extractor
### Extracts tables from PDF files (text-based or scanned/image PDFs) → Excel / CSV
**Works fully offline. No internet required.**

---
### 📋 One-time Setup
Before running, install the required packages in your terminal:
```
pip install pymupdf pytesseract Pillow openpyxl opencv-python-headless pdfplumber numpy
```
Also install **Tesseract OCR** (system binary):
- **Windows**: https://github.com/UB-Mannheim/tesseract/wiki  → install → add to PATH
- **Linux**: `sudo apt install tesseract-ocr`
- **macOS**: `brew install tesseract`

---


In [ ]:
# ============================================================
#  ✏️  CONFIGURATION  —  Edit these values before running
# ============================================================

# Path to your input PDF file
# Windows example : r"C:\Users\YourName\Documents\report.pdf"
# Linux / macOS   : "/home/yourname/documents/report.pdf"
INPUT_PDF = r"C:\Users\YourName\Documents\your_file.pdf"

# Where to save the output file (folder must exist)
# Leave as None to save in the same folder as the PDF
OUTPUT_FILE = None   # e.g. r"C:\Users\YourName\Desktop\output.xlsx"

# Output format: "xlsx"  or  "csv"
OUTPUT_FORMAT = "xlsx"

# Pages to process: "all"  or  "1,3,5"  or  "2-6"
PAGES = "all"

# OCR mode:
#   "auto"  -> auto-detect (uses OCR only for scanned/image pages)
#   "yes"   -> always use OCR  (use for scanned/photograph PDFs)
#   "no"    -> never use OCR   (use for text-selectable PDFs only)
OCR_MODE = "auto"

# Rotation correction before OCR:
#   "auto"  -> auto-detect and fix rotation
#   "yes"   -> always correct rotation  (use for sideways scans)
#   "no"    -> do not rotate
ROTATE_MODE = "auto"

# DPI for rendering scanned pages (higher = better quality, slower)
# Recommended: 300 for normal, 400-500 for small/poor quality text
DPI = 300

# [Windows only] If Tesseract is installed but not found automatically,
# set the full path here. Leave as None if Tesseract is in your PATH.
TESSERACT_PATH = None   # e.g. r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# ============================================================
print("✅ Configuration loaded.")
print(f"   Input  : {INPUT_PDF}")
print(f"   Format : {OUTPUT_FORMAT.upper()}")
print(f"   Pages  : {PAGES}")
print(f"   OCR    : {OCR_MODE}  |  Rotate: {ROTATE_MODE}  |  DPI: {DPI}")


In [ ]:
# ============================================================
#  📦  Imports & Dependency Check
# ============================================================
import sys, os, re, csv, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

missing = []
for pkg, pip_name in [
    ("fitz",        "pymupdf"),
    ("pytesseract", "pytesseract"),
    ("PIL",         "Pillow"),
    ("openpyxl",    "openpyxl"),
    ("cv2",         "opencv-python-headless"),
    ("pdfplumber",  "pdfplumber"),
    ("numpy",       "numpy"),
]:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pip_name)

if missing:
    print("❌ Missing packages — install them and restart the kernel:")
    for m in missing:
        print(f"   pip install {m}")
    raise SystemExit("Install missing packages first.")

import fitz
import pytesseract
from PIL import Image
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import numpy as np
import pdfplumber

# Set Tesseract path if provided
if TESSERACT_PATH:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

print("✅ All dependencies loaded successfully.")


In [ ]:
# ============================================================
#  🔧  Helper Functions  (run once, do not edit)
# ============================================================

def parse_pages(spec, total):
    """Parse page range string into list of 0-based indices."""
    if spec.lower() == "all":
        return list(range(total))
    pages = set()
    for part in spec.split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-", 1)
            pages.update(range(int(a) - 1, int(b)))
        else:
            pages.add(int(part) - 1)
    return sorted(p for p in pages if 0 <= p < total)


def page_has_text(fitz_page, threshold=50):
    """Return True if PDF page contains selectable text."""
    return len(fitz_page.get_text("text").strip()) > threshold


def render_page(fitz_page, dpi=300):
    """Render a PDF page to a PIL Image at the given DPI."""
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = fitz_page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def auto_rotate(pil_img):
    """Use Tesseract OSD to detect and correct image rotation."""
    try:
        osd = pytesseract.image_to_osd(pil_img, output_type=pytesseract.Output.DICT)
        angle = osd.get("rotate", 0)
        if angle:
            pil_img = pil_img.rotate(-angle, expand=True)
            print(f"      ↺ Auto-rotated {angle}°")
    except Exception:
        pass
    return pil_img


def cluster(values, gap):
    """Cluster a list of coordinate values with a proximity gap."""
    sv = sorted(set(values))
    if not sv:
        return {}
    clusters, cur = [], [sv[0]]
    for v in sv[1:]:
        if v - cur[-1] <= gap:
            cur.append(v)
        else:
            clusters.append(cur)
            cur = [v]
    clusters.append(cur)
    mapping = {}
    for idx, cl in enumerate(clusters):
        for v in cl:
            mapping[v] = idx
    return mapping


def ocr_to_grid(pil_img):
    """
    Run Tesseract OCR and reconstruct a 2D table grid
    by clustering word bounding boxes into rows and columns.
    """
    data = pytesseract.image_to_data(
        pil_img,
        output_type=pytesseract.Output.DICT,
        config="--psm 6",
    )
    words = [
        {"text": str(data["text"][i]).strip(),
         "left": data["left"][i],
         "top":  data["top"][i]}
        for i in range(len(data["text"]))
        if int(data["conf"][i]) > 10 and str(data["text"][i]).strip()
    ]
    if not words:
        return []

    row_map = cluster([w["top"]  for w in words], gap=12)
    col_map = cluster([w["left"] for w in words], gap=20)

    for w in words:
        w["r"] = row_map[w["top"]]
        w["c"] = col_map[w["left"]]

    n_rows = max(w["r"] for w in words) + 1
    n_cols = max(w["c"] for w in words) + 1
    grid = [[""] * n_cols for _ in range(n_rows)]

    for w in words:
        r, c = w["r"], w["c"]
        grid[r][c] = (grid[r][c] + " " + w["text"]).strip()

    return [row for row in grid if any(c.strip() for c in row)]


def text_to_grid(pdf_path, page_num):
    """
    Extract a table from a text-based PDF page using pdfplumber.
    Falls back to line-split text if no structured table is found.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pg = pdf.pages[page_num]
            tables = pg.extract_tables()
            if tables:
                best = max(tables, key=lambda t: sum(len(r) for r in t))
                return [[cell or "" for cell in row] for row in best]
            text = pg.extract_text()
            if text:
                return [line.split() for line in text.splitlines() if line.strip()]
    except Exception as exc:
        print(f"      ⚠ pdfplumber error: {exc}")
    return []


def write_excel(all_grids, out_path):
    """Save all extracted grids to a formatted Excel workbook."""
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    hdr_fill = PatternFill("solid", fgColor="4472C4")
    hdr_font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
    dat_font = Font(name="Calibri", size=10)
    c_align  = Alignment(horizontal="center", vertical="center", wrap_text=True)
    l_align  = Alignment(horizontal="left",   vertical="center", wrap_text=True)
    thin = Side(border_style="thin", color="BFBFBF")
    brd  = Border(left=thin, right=thin, top=thin, bottom=thin)

    for sheet_name, grid in all_grids:
        ws = wb.create_sheet(title=sheet_name[:31])
        if not grid:
            ws["A1"] = "No table data found on this page."
            continue
        for ri, row in enumerate(grid, 1):
            for ci, val in enumerate(row, 1):
                cell = ws.cell(row=ri, column=ci, value=str(val).strip() if val else "")
                cell.border = brd
                if ri == 1:
                    cell.fill, cell.font, cell.alignment = hdr_fill, hdr_font, c_align
                else:
                    cell.font, cell.alignment = dat_font, l_align
        for col_cells in ws.columns:
            ltr = col_cells[0].column_letter
            w = max((len(str(c.value)) for c in col_cells if c.value), default=8)
            ws.column_dimensions[ltr].width = min(max(w + 2, 8), 45)
        ws.freeze_panes = "A2"

    wb.save(out_path)
    print(f"  ✅ Saved Excel: {out_path}")


def write_csv(all_grids, out_path):
    """Save each page grid as a separate CSV file."""
    base   = Path(out_path).stem
    parent = Path(out_path).parent
    saved  = []
    for sheet_name, grid in all_grids:
        safe = re.sub(r"[^\w\-]", "_", sheet_name)
        fp = parent / f"{base}_{safe}.csv"
        with open(fp, "w", newline="", encoding="utf-8-sig") as f:
            csv.writer(f).writerows(
                [str(c).strip() if c else "" for c in row] for row in grid
            )
        print(f"  ✅ Saved CSV : {fp}")
        saved.append(str(fp))
    return saved

print("✅ All helper functions defined.")


In [ ]:
# ============================================================
#  🚀  Run Extraction
# ============================================================

pdf_path = Path(INPUT_PDF)

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

# Determine output path
if OUTPUT_FILE:
    out_path = Path(OUTPUT_FILE)
else:
    suffix = ".xlsx" if OUTPUT_FORMAT == "xlsx" else ".csv"
    out_path = pdf_path.with_suffix(suffix)

print("=" * 55)
print("  PDF Table Extractor")
print("=" * 55)
print(f"  Input  : {pdf_path}")
print(f"  Output : {out_path}")
print(f"  Format : {OUTPUT_FORMAT.upper()}")

doc = fitz.open(str(pdf_path))
total_pages = len(doc)
pages_to_process = parse_pages(PAGES, total_pages)

print(f"  Total pages in PDF : {total_pages}")
print(f"  Pages to process   : {[p+1 for p in pages_to_process]}")
print()

all_grids = []

for p in pages_to_process:
    label = f"Page_{p + 1}"
    print(f"  [{label}]")
    fitz_page = doc[p]

    # Decide OCR vs text mode
    if OCR_MODE == "yes":
        use_ocr = True
    elif OCR_MODE == "no":
        use_ocr = False
    else:  # auto
        use_ocr = not page_has_text(fitz_page)

    if use_ocr:
        print("    Mode   : OCR (image-based)")
        img = render_page(fitz_page, dpi=DPI)
        if ROTATE_MODE in ("yes", "auto"):
            img = auto_rotate(img)
        grid = ocr_to_grid(img)
    else:
        print("    Mode   : Text extraction")
        grid = text_to_grid(str(pdf_path), p)

    rows = len(grid)
    cols = max((len(r) for r in grid), default=0)
    print(f"    Result : {rows} rows × {cols} cols")
    all_grids.append((label, grid))

doc.close()

# Save output
if OUTPUT_FORMAT == "xlsx":
    write_excel(all_grids, str(out_path))
else:
    write_csv(all_grids, str(out_path))

print()
print("🎉 Extraction complete!")


In [ ]:
# ============================================================
#  👁️  Preview Extracted Data  (optional)
# ============================================================
# Shows first 10 rows of each extracted page in the notebook.

for sheet_name, grid in all_grids:
    if not grid:
        print(f"\n[{sheet_name}] — No data extracted.")
        continue
    print(f"\n{'='*55}")
    print(f"  {sheet_name}  |  {len(grid)} rows × {max(len(r) for r in grid)} cols")
    print(f"{'='*55}")
    for row in grid[:10]:
        print("  |  ".join(str(c).strip()[:25].ljust(25) for c in row))
    if len(grid) > 10:
        print(f"  ... ({len(grid)-10} more rows not shown)")


---
## 🗂️ Batch Mode — Process an Entire Folder of PDFs
Use the cell below to process **all PDFs in a folder** at once.
Each PDF will produce its own output file.


In [ ]:
# ============================================================
#  🗂️  BATCH CONFIG  —  Edit these values
# ============================================================

# Folder containing all your PDF files
BATCH_INPUT_FOLDER  = r"C:\Users\YourName\Documents\PDFs"

# Folder where output files will be saved (must exist)
BATCH_OUTPUT_FOLDER = r"C:\Users\YourName\Documents\Output"

# Same options as single-file mode
BATCH_FORMAT      = "xlsx"    # "xlsx" or "csv"
BATCH_PAGES       = "all"
BATCH_OCR_MODE    = "auto"    # "auto" / "yes" / "no"
BATCH_ROTATE_MODE = "auto"    # "auto" / "yes" / "no"
BATCH_DPI         = 300

# ============================================================
import glob

pdf_files = sorted(glob.glob(os.path.join(BATCH_INPUT_FOLDER, "*.pdf")))
print(f"Found {len(pdf_files)} PDF(s) in {BATCH_INPUT_FOLDER}")

for pdf_file in pdf_files:
    pdf_path = Path(pdf_file)
    suffix   = ".xlsx" if BATCH_FORMAT == "xlsx" else ".csv"
    out_path = Path(BATCH_OUTPUT_FOLDER) / pdf_path.with_suffix(suffix).name

    print(f"\n{'─'*50}")
    print(f"Processing: {pdf_path.name}")

    try:
        doc   = fitz.open(str(pdf_path))
        pages = parse_pages(BATCH_PAGES, len(doc))
        grids = []

        for p in pages:
            fitz_page = doc[p]
            label     = f"Page_{p+1}"

            if BATCH_OCR_MODE == "yes":
                use_ocr = True
            elif BATCH_OCR_MODE == "no":
                use_ocr = False
            else:
                use_ocr = not page_has_text(fitz_page)

            if use_ocr:
                img = render_page(fitz_page, dpi=BATCH_DPI)
                if BATCH_ROTATE_MODE in ("yes", "auto"):
                    img = auto_rotate(img)
                grid = ocr_to_grid(img)
            else:
                grid = text_to_grid(str(pdf_path), p)

            grids.append((label, grid))
            rows = len(grid)
            cols = max((len(r) for r in grid), default=0)
            print(f"  [{label}] {rows}r × {cols}c")

        doc.close()

        if BATCH_FORMAT == "xlsx":
            write_excel(grids, str(out_path))
        else:
            write_csv(grids, str(out_path))

    except Exception as exc:
        print(f"  ❌ ERROR processing {pdf_path.name}: {exc}")
        continue

print("\n✅ Batch processing complete!")
